# Cattle Weight — Step D: MobileNetV3 (on-device)
Attach `cow-data-prep` dataset. Runs in parallel with B and C.
Publish output as `cow-mobile`.

In [ ]:
import subprocess, sys
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)
pip('albumentations', 'scikit-image', 'rembg[cpu]')


In [ ]:
import os, json, re, warnings, gc, math, time, tarfile, concurrent.futures as cf
from pathlib import Path
from collections import defaultdict
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import albumentations as A
warnings.filterwarnings('ignore')

print('TF  :', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
tf.keras.mixed_precision.set_global_policy('mixed_float16')

SEED = 42
OUT  = Path('/kaggle/working')
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
# ── Load shared data-prep outputs ─────────────────────────────────────────────
# Auto-detect the cow-data-prep dataset path (Kaggle mounts vary by account).
def _find_data_prep():
    for p in sorted(Path('/kaggle/input').rglob('norm_state.json')):
        return str(p.parent)
    raise FileNotFoundError(
        "norm_state.json not found under /kaggle/input. "
        "Make sure you attached the cow-data-prep dataset to this kernel.")

DATA_PREP = _find_data_prep()
print(f'Data prep found at: {DATA_PREP}')

state       = json.load(open(f'{DATA_PREP}/norm_state.json'))
LOG_MEAN    = state['log_mean'];    LOG_STD     = state['log_std']
WEIGHT_MEAN = state['weight_mean']; WEIGHT_STD  = state['weight_std']
print(f'Norm state: log_mean={LOG_MEAN:.4f}  log_std={LOG_STD:.4f}')

train_df = pd.read_csv(f'{DATA_PREP}/train_split.csv')
val_df   = pd.read_csv(f'{DATA_PREP}/val_split.csv')
test_df  = pd.read_csv(f'{DATA_PREP}/test_split.csv')
for df in [train_df, val_df, test_df]:
    df['bbox'] = df['bbox'].apply(
        lambda x: json.loads(x) if pd.notna(x) and str(x) not in ('nan','null','') else None)
print(f'Split loaded: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')
# Remap full_path to actual mount location in this kernel
def _find_dataset_root(sample_path):
    slug = Path(sample_path).parts[3]
    for root, dirs, _ in os.walk('/kaggle/input'):
        if slug in dirs:
            return os.path.join(root, slug)
    return None

def _remap_paths(df):
    sample = df['full_path'].dropna().iloc[0]
    if Path(sample).exists():
        return df
    old_root = '/'.join(Path(sample).parts[:4])
    new_root = _find_dataset_root(sample)
    if new_root:
        df = df.copy()
        df['full_path'] = df['full_path'].str.replace(old_root, new_root, regex=False)
        print(f'Remapped: {old_root}')
        print(f'      ->  {new_root}')
    else:
        print(f'[WARN] Could not find dataset root for: {sample}')
    return df

train_df = _remap_paths(train_df)
val_df   = _remap_paths(val_df)
test_df  = _remap_paths(test_df)
_s = train_df['full_path'].dropna().iloc[0]
print(f'Path check: {"OK" if Path(_s).exists() else "STILL BROKEN"} — {_s[:80]}')


MASK_DIR = Path('/kaggle/working/masks')
MASK_DIR.mkdir(exist_ok=True)
# Search recursively so the file is found regardless of dataset nesting depth
_tar_hits = (list(Path('/kaggle/input').rglob('masks.tar.gz'))
             or list(Path('/kaggle/input').rglob('masks.tar')))
if _tar_hits:
    _tar = _tar_hits[0]
    print(f'Extracting masks from {_tar} ...')
    with tarfile.open(_tar) as t:
        t.extractall('/kaggle/working/')
    print(f'  {sum(1 for _ in MASK_DIR.glob("*.png")):,} masks ready')
else:
    _mask_dirs = [p for p in Path('/kaggle/input').rglob('masks') if p.is_dir()]
    if _mask_dirs:
        import shutil
        _all_pngs = [p for _md in _mask_dirs for p in _md.rglob('*.png')]
        print(f'  Found {len(_all_pngs)} PNG masks under {_mask_dirs[0]}')
        for _png in _all_pngs:
            shutil.copy2(str(_png), str(MASK_DIR / _png.name))

_mask_count = sum(1 for _ in MASK_DIR.glob('*.png'))
if _mask_count > 0:
    print(f'  {_mask_count:,} masks ready')
else:
    print('Masks empty/missing — generating with rembg (this runs once per session)...')
    try:
        from rembg import remove, new_session
        _sess = new_session('u2net')
        _all = (pd.concat([train_df, val_df, test_df], ignore_index=True)
                  [['full_path', 'file_name']].drop_duplicates())
        def _gen_mask(row):
            try:
                stem = Path(row['file_name']).stem
                out  = MASK_DIR / f'{stem}.png'
                if out.exists(): return
                mask = remove(Image.open(row['full_path']).convert('RGB'),
                              session=_sess, only_mask=True)
                mask.save(out)
            except Exception: pass
        for _row in tqdm(_all.to_dict('records'), desc='rembg masks'):
            _gen_mask(_row)
        print(f'  Generated {sum(1 for _ in MASK_DIR.glob("*.png")):,} masks')
    except BaseException as _e:
        print(f'[WARN] rembg failed ({_e}) — body-geometry features will be zeros')



In [ ]:
# ── Augmentation + data pipeline ──────────────────────────────────────────────
IMG_SIZE = 224
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

train_aug = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0), ratio=(0.75, 1.33), p=0.5),
    A.LongestMaxSize(IMG_SIZE),
    A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(0.25, 0.25, p=0.6),
    A.HueSaturationValue(15, 30, 15, p=0.5),
    A.RandomGamma(gamma_limit=(70, 130), p=0.3),
    A.GaussNoise(var_limit=(10, 80), p=0.35),
    A.Blur(blur_limit=5, p=0.25),
    A.ShiftScaleRotate(0.08, 0.15, 15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
    A.CoarseDropout(max_holes=6, max_height=48, max_width=48, p=0.3),
    A.CLAHE(p=0.3),
    A.Normalize(_MEAN, _STD),
])
val_aug = A.Compose([
    A.LongestMaxSize(IMG_SIZE),
    A.PadIfNeeded(IMG_SIZE, IMG_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Normalize(_MEAN, _STD),
])
AUX_DIM = 16

class GeM(layers.Layer):
    def __init__(self, p=3.0, eps=1e-6, **kw):
        super().__init__(**kw)
        self._p  = tf.Variable(p, trainable=True, dtype=tf.float32, name='gem_p')
        self.eps = eps
    def call(self, x):
        x = tf.cast(x, tf.float32)
        x = tf.pow(tf.maximum(x, self.eps), self._p)
        x = tf.reduce_mean(x, axis=[1, 2])
        return tf.cast(tf.pow(x, 1.0 / self._p), self.compute_dtype)
    def get_config(self):
        return {**super().get_config(), 'p': float(self._p.numpy()), 'eps': self.eps}

def _extract_from_array(mask_crop):
    try:
        H, W = mask_crop.shape[:2]
        _, binary = cv2.threshold(mask_crop, 127, 255, cv2.THRESH_BINARY)
        total_px = H * W
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return np.zeros(14, np.float32)
        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True) + 1e-6
        hull_area = cv2.contourArea(cv2.convexHull(cnt)) + 1e-6
        x, y, bw, bh = cv2.boundingRect(cnt)
        M = cv2.moments(cnt)
        cx = M['m10'] / (M['m00'] + 1e-6) / W
        cy = M['m01'] / (M['m00'] + 1e-6) / H
        roi = cv2.resize(binary[y:y+bh, x:x+bw], (128, 128)) > 127
        try:
            from skimage.morphology import skeletonize
            skel_len = float(skeletonize(roi).sum()) / (128 * 128)
        except Exception:
            skel_len = 0.0
        diag = np.sqrt(W**2 + H**2) + 1e-6
        ma, mi, angle = (0.0, 0.0, 0.0)
        if len(cnt) >= 5:
            _, (ma, mi), angle = cv2.fitEllipse(cnt)
        return np.clip(np.array([
            area / total_px, bw / (bh + 1e-6), bw / W, bh / H,
            area / hull_area, (4 * np.pi * area) / (perimeter ** 2),
            area / (bw * bh + 1e-6), skel_len,
            ma / diag, mi / diag,
            np.sin(np.deg2rad(angle)), np.cos(np.deg2rad(angle)),
            cx, cy,
        ], np.float32), 0.0, 1.0)
    except Exception:
        return np.zeros(14, np.float32)

def load_sample(row, augment=True):
    _entry = IMG_CACHE.get(row.get('full_path', ''))
    if _entry is not None:
        img, body_feats = _entry[0].copy(), _entry[1].copy()
    else:
        img = np.zeros((CACHE_SIZE, CACHE_SIZE, 3), np.uint8)
        body_feats = np.zeros(14, np.float32)
    img = (train_aug if augment else val_aug)(image=img)['image'].astype(np.float32)
    age   = row.get('age', float('nan'))
    age_n = float(age) / 10.0 if pd.notna(age) else 0.0
    sex_b = 1.0 if str(row.get('sex', '')).lower() in ('m', 'male', 'bull') else 0.0
    aux   = np.concatenate([[age_n, sex_b], body_feats]).astype(np.float32)
    log_w = np.log(float(row['weight_kg']))
    if augment:
        log_w += np.random.normal(0.0, 0.02)
    return img, aux, np.float32((log_w - LOG_MEAN) / LOG_STD)

def _mixup_batch(inputs, labels):
    imgs = inputs['image']; auxs = inputs['aux']
    B    = tf.shape(imgs)[0]
    lam  = tf.random.uniform((), 0.2, 0.8)
    idx  = tf.random.shuffle(tf.range(B))
    return ({'image': lam*imgs + (1-lam)*tf.gather(imgs, idx),
             'aux':   lam*auxs + (1-lam)*tf.gather(auxs, idx)},
            lam*labels + (1-lam)*tf.gather(labels, idx))

def make_tf_dataset(df_split, batch_size=128, shuffle=True, augment=True):
    rows = df_split.to_dict('records')

    def _load(idx):
        return load_sample(rows[int(idx)], augment)

    def tf_load(idx):
        img, aux, label = tf.py_function(_load, [idx], [tf.float32, tf.float32, tf.float32])
        img.set_shape((IMG_SIZE, IMG_SIZE, 3))
        aux.set_shape((AUX_DIM,))
        label.set_shape(())
        return {'image': img, 'aux': aux}, label

    ds = tf.data.Dataset.from_tensor_slices(np.arange(len(rows)))
    if shuffle:
        ds = ds.shuffle(len(rows), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    if not augment:
        ds = ds.cache()
    ds = ds.batch(batch_size)
    if augment:
        ds = ds.map(_mixup_batch, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def mape_metric(y_true, y_pred):
    yt = tf.exp(y_true * LOG_STD + LOG_MEAN)
    yp = tf.exp(y_pred * LOG_STD + LOG_MEAN)
    return tf.reduce_mean(tf.abs(yt - yp) / (yt + 1e-6)) * 100.0

def mae_kg_metric(y_true, y_pred):
    yt = tf.exp(y_true * LOG_STD + LOG_MEAN)
    yp = tf.exp(y_pred * LOG_STD + LOG_MEAN)
    return tf.reduce_mean(tf.abs(yt - yp))

print(f'Pipeline ready  IMG_SIZE={IMG_SIZE}  AUX_DIM={AUX_DIM}')

# ── Preload: bbox crop + mask apply + resize 256×256 + body features ─────────
# Stores (img_uint8, body_feats) per path. Only augmentation runs per step.
CACHE_SIZE = 256  # > IMG_SIZE so RandomResizedCrop has room
IMG_CACHE  = {}   # path -> (img [CACHE_SIZE,CACHE_SIZE,3] uint8, feats [14])

def _preload_row(row):
    path = row.get('full_path', '')
    if not path or path in IMG_CACHE:
        return
    img = cv2.imread(path)
    if img is None:
        IMG_CACHE[path] = (np.zeros((CACHE_SIZE, CACHE_SIZE, 3), np.uint8),
                           np.zeros(14, np.float32))
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H0, W0 = img.shape[:2]
    crop_coords = None
    bbox = row.get('bbox')
    if isinstance(bbox, list):
        try:
            x, y, bw, bh = [max(0, int(v)) for v in bbox]
            pad = int(max(bw, bh) * 0.12)
            x1=max(0,x-pad); y1=max(0,y-pad)
            x2=min(W0,x+bw+pad); y2=min(H0,y+bh+pad)
            if x2 > x1 and y2 > y1:
                crop_coords = (x1, y1, x2, y2)
                img = img[y1:y2, x1:x2]
        except Exception:
            pass
    stem = Path(row.get('file_name', '')).stem
    mask_path = MASK_DIR / f'{stem}.png'
    body_feats = np.zeros(14, np.float32)
    if mask_path.exists():
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            mc = mask[crop_coords[1]:crop_coords[3],
                      crop_coords[0]:crop_coords[2]] if crop_coords else mask
            body_feats = _extract_from_array(mc)
            if mc.shape[:2] == img.shape[:2]:
                img = (img * (mc[:,:,None] / 255.0)).astype(np.uint8)
    # Resize longest side to CACHE_SIZE, pad to square
    h, w = img.shape[:2]
    s = CACHE_SIZE / max(h, w, 1)
    nh, nw = max(1, int(h*s)), max(1, int(w*s))
    img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    img = cv2.copyMakeBorder(img, 0, CACHE_SIZE-nh, 0, CACHE_SIZE-nw,
                             cv2.BORDER_CONSTANT, value=0)
    IMG_CACHE[path] = (img.astype(np.uint8), body_feats)

print('Preloading images (crop+mask+resize to 256×256)...')
_all_rows = (pd.concat([train_df, val_df, test_df], ignore_index=True)
               .drop_duplicates('full_path').to_dict('records'))
for _row in tqdm(_all_rows, desc='preloading'):
    _preload_row(_row)
_cache_mb = sum(img.nbytes for img,_ in IMG_CACHE.values()) / 1e6
print(f'Cached {len(IMG_CACHE):,} images ({_cache_mb:.0f} MB)')


In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────
BATCH_SIZE = 128

train_ds = make_tf_dataset(train_df, BATCH_SIZE, shuffle=True,  augment=True)
val_ds   = make_tf_dataset(val_df,   BATCH_SIZE, shuffle=False, augment=False)
test_ds  = make_tf_dataset(test_df,  BATCH_SIZE, shuffle=False, augment=False)
print(f'Datasets ready  batch={BATCH_SIZE}  '
      f'train={math.ceil(len(train_df)/BATCH_SIZE)} batches')

class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, total_steps, warmup_steps):
        self.base_lr = base_lr; self.total_steps = total_steps
        self.warmup_steps = warmup_steps
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        w = self.base_lr * (step / self.warmup_steps)
        c = self.base_lr * 0.5 * (1 + tf.cos(
            math.pi * (step - self.warmup_steps)
            / (self.total_steps - self.warmup_steps)))
        return tf.where(step < self.warmup_steps, w, c)
    def get_config(self):
        return dict(base_lr=self.base_lr,
                    total_steps=self.total_steps,
                    warmup_steps=self.warmup_steps)

def make_callbacks(name, patience=5):
    return [
        keras.callbacks.ModelCheckpoint(
            str(OUT / f'{name}_best.keras'),
            monitor='val_mape_metric', mode='min',
            save_best_only=True, verbose=0),
        keras.callbacks.EarlyStopping(
            monitor='val_mape_metric', mode='min',
            patience=patience, restore_best_weights=True),
        keras.callbacks.CSVLogger(str(OUT / f'{name}_log.csv')),
    ]

def train_phase(model, train_ds, val_ds, epochs, lr, phase, patience=4):
    steps = epochs * math.ceil(len(train_df) / BATCH_SIZE)
    sched = WarmupCosine(lr, steps, max(1, int(0.05 * steps)))
    model.compile(
        optimizer=keras.optimizers.AdamW(sched, weight_decay=5e-4, clipnorm=1.0),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=[mape_metric, mae_kg_metric])
    print(f'\n=== {phase} ===')
    return model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                     callbacks=make_callbacks(model.name, patience), verbose=1)

def save_results(name, results):
    json.dump({'loss': float(results[0]), 'mape': float(results[1]),
               'mae_kg': float(results[2])},
              open(OUT / f'{name}_results.json', 'w'))
    print(f'Saved {name}_results.json  MAPE={results[1]:.2f}%  MAE={results[2]:.1f} kg')


In [ ]:
# ── MobileNetV3-Large (224px) ─────────────────────────────────────────────────
def build_mobilenet():
    from tensorflow.keras.applications import MobileNetV3Large
    inp_img = layers.Input((IMG_SIZE, IMG_SIZE, 3), name='image')
    inp_aux = layers.Input((AUX_DIM,), name='aux')
    base = MobileNetV3Large(include_top=False, weights='imagenet', minimalistic=False,
                            include_preprocessing=False)
    base.trainable = False
    x = GeM(name='gem')(base(inp_img))
    x = layers.Dropout(0.3)(x)
    a = layers.Dense(64, activation='swish')(inp_aux)
    a = layers.BatchNormalization()(a)
    x = layers.Concatenate()([x, a])
    x = layers.BatchNormalization()(x)
    for u in (256, 128):
        x = layers.Dense(u, activation='swish')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
    return Model([inp_img, inp_aux], layers.Dense(1, dtype='float32', name='weight')(x),
                 name='mobilenetv3')

model = build_mobilenet()
model.summary(line_length=80, expand_nested=False)

hist = train_phase(model, train_ds, val_ds, epochs=25, lr=1e-3,
                   phase='Mobile frozen', patience=8)

results = model.evaluate(test_ds, verbose=0)
print(f'Mobile Test  MAPE={results[1]:.2f}%  MAE={results[2]:.1f} kg')
save_results('mobilenetv3', results)
model.save(str(OUT / 'mobilenetv3_final.keras'))
print('Done. Publish this kernel output as a Kaggle dataset (e.g. cow-mobile).')
